# Blooming Peccioli — Build Scenarios
Legge gli shapefile di fermate e grafo stradale e produce `weekday.json`, `weekend.json`, `eventday.json`
da copiare in `data/scenarios/` nel repo GitHub.

In [93]:
# ── DIPENDENZE ──────────────────────────────────────────────────────────────
# pip install geopandas networkx shapely numpy pyproj

import geopandas as gpd
import networkx as nx
import numpy as np
import json, math, random
from shapely.geometry import Point
from pyproj import Transformer

In [ ]:
# ── PARAMETRI — modifica questi percorsi ────────────────────────────────────

STOPS_SHP   = "C:/Users/m.kroetz/OneDrive - Systematica/Valdera/06_WORK/GIS/fermate bacini/fermate_bacini_snapped.shp"      # shapefile fermate (CRS 4326)
NETWORK_SHP = "C:/Users/m.kroetz/OneDrive - Systematica/Valdera/06_WORK/Database/GIS/osm_network/edges_filtered.shp"    # shapefile grafo strade (archi, CRS 4326)

OUT_DIR = r"C:\Users\m.kroetz\OneDrive - Systematica\Valdera\repo\blooming_peccioli\data\scenarios"      # dove scrivere i JSON

SPEED_MS = 25 * 1000 / 3600          # 25 km/h in m/s
RANDOM_SEED = 42                     # ripetibilità delle attivazioni on-demand
PONTEDERA_WAIT = 600 

# Orari in secondi da mezzanotte
WEEKDAY_PEAK_AM   = 8  * 3600   # 08:00
WEEKDAY_PEAK_PM   = 18 * 3600   # 18:00
WEEKEND_PEAK_AM   = 10 * 3600   # 10:00
WEEKEND_PEAK_PM   = 20 * 3600   # 20:00
EVENT_TRIP1       = 12 * 3600   # 12:00 peccioli_1 → belvedere_5
EVENT_TRIP2       = 17 * 3600   # 17:00 belvedere_5 → peccioli_1

# Colori veicoli
COLORS = {
    "jolly":     "#ffffff",
    "peccioli":   "#FFC000",
    "forcoli":    "#41CE69",
    "fabbrica":   "#BC7CFF",
    "belvedere":  "#0875F9",
    "rosa":       "#FD6BC9",
    "ghizzano":   "#FF9900",
}

In [ ]:
# ── CARICA SHAPEFILE ────────────────────────────────────────────────────────

stops_gdf   = gpd.read_file(STOPS_SHP).to_crs(epsg=4326)
network_gdf = gpd.read_file(NETWORK_SHP).to_crs(epsg=4326)

print(f"Fermate: {len(stops_gdf)} | Archi rete: {len(network_gdf)}")
print("Colonne fermate:", list(stops_gdf.columns))
print("Colonne rete:",    list(network_gdf.columns))
stops_gdf.head()

In [ ]:
# ── COSTRUISCI GRAFO NETWORKX ────────────────────────────────────────────────
# Archi in metri (lunghezza calcolata dalla geometria in proiezione metrica)

# Convertiamo in UTM 32N per avere distanze in metri
net_utm = network_gdf.to_crs(epsg=32632)

G = nx.DiGraph()

for _, row in net_utm.iterrows():
    u = row["u"]
    v = row["v"]
    length = row.geometry.length  # metri
    G.add_edge(u, v, length=length)
    if not row.get("oneway", False):  # se non è senso unico aggiungiamo il contrario
        G.add_edge(v, u, length=length)

print(f"Grafo: {G.number_of_nodes()} nodi, {G.number_of_edges()} archi")

In [ ]:
# ── AGGIUNGI COORDINATE AI NODI ──────────────────────────────────────────────
# Costruiamo un dict node_id → (lat, lon) dai nodi toccati dagli archi.
# Usiamo le coordinate dei punti iniziali/finali della geometria degli archi.

node_coords_utm = {}  # node_id → (easting, northing) in UTM 32N

for _, row in net_utm.iterrows():
    coords = list(row.geometry.coords)
    node_coords_utm[row["u"]] = coords[0]   # primo punto dell'arco
    node_coords_utm[row["v"]] = coords[-1]  # ultimo punto dell'arco

# Convertiamo in lat/lon per i waypoint del frontend
transformer = Transformer.from_crs("epsg:32632", "epsg:4326", always_xy=True)

node_latlon = {}  # node_id → (lat, lon)
for nid, (x, y) in node_coords_utm.items():
    lon, lat = transformer.transform(x, y)
    node_latlon[nid] = (lat, lon)

print(f"Coordinate nodi: {len(node_latlon)}")

In [ ]:
# ── FUNZIONI HELPER ──────────────────────────────────────────────────────────

def find_nearest_node(lat, lon, node_latlon_dict):
    """Trova il nodo del grafo più vicino a una coordinata."""
    best_node = None
    best_dist = float("inf")
    for nid, (nlat, nlon) in node_latlon_dict.items():
        d = math.hypot(lat - nlat, lon - nlon)
        if d < best_dist:
            best_dist = d
            best_node = nid
    return best_node


def route_time_and_waypoints(G, node_latlon, start_node, end_node, speed_ms):
    """
    Calcola shortest path sul grafo tra due nodi.
    Restituisce (travel_time_seconds, [(lat, lon), ...]) lungo il percorso.
    """
    try:
        path = nx.shortest_path(G, start_node, end_node, weight="length")
    except nx.NetworkXNoPath:
        print(f"  ⚠️  Nessun percorso tra {start_node} e {end_node}, uso linea retta")
        # fallback: linea retta
        c0 = node_latlon[start_node]
        c1 = node_latlon[end_node]
        dist = math.hypot(
            (c0[0]-c1[0]) * 111320,
            (c0[1]-c1[1]) * 111320 * math.cos(math.radians(c0[0]))
        )
        return dist / speed_ms, [c0, c1]

    coords = [node_latlon[n] for n in path]
    total_length = sum(
        G[path[i]][path[i+1]]["length"] for i in range(len(path)-1)
    )
    travel_time = total_length / speed_ms
    return travel_time, coords


def build_timed_waypoints(G, node_latlon, stop_list_latlon, t_start, speed_ms):
    """
    Waypoint con t proporzionale alla distanza cumulativa lungo il percorso.
    Evita variazioni di velocità apparente causate da nodi irregolarmente spaziati.
    """
    waypoints = []
    t = t_start
    prev_node = find_nearest_node(*stop_list_latlon[0], node_latlon)
    waypoints.append({"t": int(t), "lat": stop_list_latlon[0][0], "lng": stop_list_latlon[0][1]})

    for stop_lat, stop_lon in stop_list_latlon[1:]:
        curr_node = find_nearest_node(stop_lat, stop_lon, node_latlon)
        travel_time, path_coords = route_time_and_waypoints(G, node_latlon, prev_node, curr_node, speed_ms)

        # Calcola distanze cumulative tra nodi consecutivi
        cum_dists = [0.0]
        for i in range(1, len(path_coords)):
            dlat = (path_coords[i][0] - path_coords[i-1][0]) * 111320
            dlng = (path_coords[i][1] - path_coords[i-1][1]) * 111320 * math.cos(math.radians(path_coords[i-1][0]))
            cum_dists.append(cum_dists[-1] + math.hypot(dlat, dlng))

        total_dist = cum_dists[-1] if cum_dists[-1] > 0 else 1

        for i in range(1, len(path_coords)):
            frac = cum_dists[i] / total_dist          # proporzionale alla distanza
            t_wp = t + travel_time * frac
            waypoints.append({"t": int(t_wp), "lat": path_coords[i][0], "lng": path_coords[i][1]})

        t += travel_time
        prev_node = curr_node

    return waypoints, t

In [ ]:
# ── ORGANIZZA FERMATE PER BACINO ─────────────────────────────────────────────

# Dal campo 'id' tipo 'rosa_1', 'belvedere_5', 'peccioli_1' ecc.
# Assumiamo colonna 'id' e 'po_andata' nel shapefile.

def bacino_name(stop_id):
    return "_".join(stop_id.split("_")[:-1])  # 'rosa_1' → 'rosa'

def stop_number(stop_id):
    return int(stop_id.split("_")[-1])  # 'rosa_1' → 1

stops_gdf["bacino"]  = stops_gdf["id"].apply(bacino_name)
stops_gdf["stop_num"] = stops_gdf["id"].apply(stop_number)
stops_gdf["lat"] = stops_gdf.geometry.y
stops_gdf["lon"] = stops_gdf.geometry.x

# Dizionario bacino → fermate ordinate per po_andata
bacini = {}
for bacino, group in stops_gdf.groupby("bacino"):
    sorted_stops = group.sort_values("po_andata").reset_index(drop=True)
    bacini[bacino] = list(zip(sorted_stops["lat"], sorted_stops["lon"], sorted_stops["id"]))
    
# pila_1 appartiene logicamente al bacino forcoli
if "pila" in bacini:
    forcoli_stops = bacini.get("forcoli", []) + bacini["pila"]
    forcoli_stops = sorted(
        forcoli_stops,
        key=lambda s: stops_gdf[stops_gdf["id"] == s[2]]["po_andata"].values[0]
    )
    bacini["forcoli"] = forcoli_stops
    del bacini["pila"]    

print("Bacini trovati:", list(bacini.keys()))

# Fermata hub principale
hub = stops_gdf[stops_gdf["id"] == "peccioli_1"].iloc[0]
HUB = (hub["lat"], hub["lon"])
print(f"Hub peccioli_1: {HUB}")

# Fermata belvedere_5 per giornata evento
bel5 = stops_gdf[stops_gdf["id"] == "belvedere_4"].iloc[0]
BEL5 = (bel5["lat"], bel5["lon"])
print(f"Belvedere_5: {BEL5}")

In [ ]:
# ── FERMATE SCHEDULED SERVICE ────────────────────────────────────────────────
# Ordine andata: peccioli_1 → pila_1 → forcoli_1 → pontedera_2 → pontedera_1
# Ordine ritorno: inverso

SCHEDULED_ORDER_IDS = ["peccioli_1", "peccioli_8", "pila_1", "forcoli_1", "pontedera_2", "pontedera_1"]

scheduled_coords = []
for sid in SCHEDULED_ORDER_IDS:
    row = stops_gdf[stops_gdf["id"] == sid].iloc[0]
    scheduled_coords.append((row["lat"], row["lon"]))

print("Fermate scheduled:", SCHEDULED_ORDER_IDS)

In [ ]:
# ── BACINI ON-DEMAND (esclusi pontedera, pila, scheduled-only) ───────────────

ON_DEMAND_BACINI = ["peccioli", "forcoli", "fabbrica", "belvedere", "rosa", "ghizzano"]

# Il punto 'più lontano' = fermata con po_andata massimo nel bacino
def get_furthest_stop(bacino_name):
    stops = bacini[bacino_name]
    return stops[-1][:2]  # (lat, lon) dell'ultima in po_andata order

for b in ON_DEMAND_BACINI:
    if b in bacini:
        print(f"{b}: {len(bacini[b])} fermate, più lontana → {bacini[b][-1][2]}")
    else:
        print(f"⚠️  bacino '{b}' NON trovato nello shapefile — controlla il campo 'id'")

In [ ]:
# ── FUNZIONE: costruisce la timeline di un veicolo scheduled ─────────────────
# Due veicoli: A parte da peccioli_1, B parte da pontedera_1, in direzioni opposte.
# Entrambi fanno andata+ritorno nella stessa corsa peak.

def build_scheduled_vehicle(vehicle_id, label, depart_time, stops_latlon, reverse=False):
    """
    reverse=False → andata (peccioli→pontedera)
    reverse=True  → ritorno (pontedera→peccioli)
    """
    route = list(reversed(stops_latlon)) if reverse else stops_latlon
    waypoints, t_end = build_timed_waypoints(G, node_latlon, route, depart_time, SPEED_MS)
    return {
        "id": vehicle_id,
        "label": label,
        "color": COLORS["scheduled"],
        "waypoints": waypoints
    }, t_end

print("Funzione build_scheduled_vehicle pronta")

In [ ]:
# # ── FUNZIONE: costruisce la timeline di un veicolo on-demand ─────────────────
# #
# # Struttura giornata (weekday esempio):
# #   1. Mattina: hub → furthest (arriva in tempo per peak_am)
# #   2. On-demand intermedio: giro random tra fermate del bacino
# #   3. Sera: rientro all'hub (quando scheduled arriva) → furthest

# def build_ondemand_vehicle(bacino, peak_am, peak_pm, seed_offset=0):
#     rng = random.Random(RANDOM_SEED + seed_offset)
#     stops  = bacini[bacino]
#     furthest   = stops[-1][:2]   # (lat, lon) fermata più lontana
#     all_latlon = [s[:2] for s in stops]

#     waypoints = []

#     # Nodi chiave
#     node_hub  = find_nearest_node(*HUB,      node_latlon)
#     node_far  = find_nearest_node(*furthest, node_latlon)

#     # Tempi di percorrenza base
#     t_hub_to_far, _ = route_time_and_waypoints(G, node_latlon, node_hub, node_far, SPEED_MS)
#     t_far_to_hub, _ = route_time_and_waypoints(G, node_latlon, node_far, node_hub, SPEED_MS)

#     # ── 1. MATTINA: hub → furthest → hub, rientro entro peak_am ──────────────
#     # Il veicolo deve essere tornato a peccioli_1 prima che parta lo scheduled
#     t_depart_morning = peak_am - t_hub_to_far - t_far_to_hub - 300  # 5 min di margine

#     wps_to_far, t_at_far = build_timed_waypoints(
#         G, node_latlon, [HUB, furthest], t_depart_morning, SPEED_MS
#     )
#     waypoints.extend(wps_to_far)

#     wps_back_am, t_at_hub_am = build_timed_waypoints(
#         G, node_latlon, [furthest, HUB], t_at_far, SPEED_MS
#     )
#     waypoints.extend(wps_back_am[1:])

#     # ── 2. ON-DEMAND MATTINA: giri nel bacino dopo il rientro ────────────────
#     # Calcola quando lo scheduled arriva all'hub la sera
#     node_pont = find_nearest_node(*scheduled_coords[-1], node_latlon)
#     t_sched_return, _ = route_time_and_waypoints(G, node_latlon, node_pont, node_hub, SPEED_MS)
#     t_scheduled_at_hub = peak_pm + t_sched_return  # ora in cui scheduled arriva a peccioli_1

#     t_current = t_at_hub_am

#     def t_to_hub_from(lat, lng):
#         n = find_nearest_node(lat, lng, node_latlon)
#         t, _ = route_time_and_waypoints(G, node_latlon, n, node_hub, SPEED_MS)
#         return t

#     # On-demand diurno: continua fino a ~1h prima del rientro serale
#     t_limit_day = t_scheduled_at_hub - t_hub_to_far - t_far_to_hub - 1800

#     while t_current < t_limit_day:
#         wait = rng.randint(120, 300)   # 2-5 min tra una corsa e l'altra
#         t_current += wait

#         last = waypoints[-1]
#         if t_current + t_to_hub_from(last["lat"], last["lng"]) >= t_limit_day:
#             break

#         n_sel = rng.randint(2, min(4, len(all_latlon)))
#         selected_idx = sorted(rng.sample(range(len(all_latlon)), n_sel))
#         selected = [all_latlon[i] for i in selected_idx]
#         route = [(last["lat"], last["lng"])] + selected

#         test_wps, t_end_run = build_timed_waypoints(G, node_latlon, route, t_current, SPEED_MS)
#         if t_end_run + t_to_hub_from(*selected[-1]) >= t_limit_day:
#             break

#         waypoints.extend(test_wps[1:])
#         t_current = t_end_run

#     # ── 3. RIENTRO HUB per incontro con scheduled serale ─────────────────────
#     last = waypoints[-1]
#     t_depart_to_hub = t_scheduled_at_hub - t_to_hub_from(last["lat"], last["lng"])
#     if t_depart_to_hub > t_current:
#         waypoints.append({"t": int(t_depart_to_hub), "lat": last["lat"], "lng": last["lng"]})

#     wps_to_hub_eve, _ = build_timed_waypoints(
#         G, node_latlon, [(last["lat"], last["lng"]), HUB], t_depart_to_hub, SPEED_MS
#     )
#     waypoints.extend(wps_to_hub_eve[1:])

#     # ── 4. SERA: hub → furthest → hub ────────────────────────────────────────
#     wps_to_far_eve, t_at_far_eve = build_timed_waypoints(
#         G, node_latlon, [HUB, furthest], t_scheduled_at_hub, SPEED_MS
#     )
#     waypoints.extend(wps_to_far_eve[1:])

#     wps_back_eve, t_back_hub_eve = build_timed_waypoints(
#         G, node_latlon, [furthest, HUB], t_at_far_eve, SPEED_MS
#     )
#     waypoints.extend(wps_back_eve[1:])

#     # ── 5. ON-DEMAND SERALE: giri fino a mezzanotte ──────────────────────────
#     t_current = t_back_hub_eve
#     t_limit_night = 86400 - 1200   # fine giornata con 20 min di margine

#     while t_current < t_limit_night:
#         wait = rng.randint(120, 300)
#         t_current += wait
#         if t_current >= t_limit_night:
#             break

#         last = waypoints[-1]
#         n_sel = rng.randint(2, min(4, len(all_latlon)))
#         selected_idx = sorted(rng.sample(range(len(all_latlon)), n_sel))
#         selected = [all_latlon[i] for i in selected_idx]
#         route = [(last["lat"], last["lng"])] + selected

#         test_wps, t_end_run = build_timed_waypoints(G, node_latlon, route, t_current, SPEED_MS)
#         if t_end_run + t_to_hub_from(*selected[-1]) >= t_limit_night:
#             break

#         waypoints.extend(test_wps[1:])
#         t_current = t_end_run

#     # ── 6. RIENTRO FINALE A PECCIOLI_1 ───────────────────────────────────────
#     last = waypoints[-1]
#     wps_final, _ = build_timed_waypoints(
#         G, node_latlon, [(last["lat"], last["lng"]), HUB], t_current + 300, SPEED_MS
#     )
#     waypoints.extend(wps_final[1:])

#     return {
#         "id": bacino,
#         "label": bacino.capitalize(),
#         "color": COLORS[bacino],
#         "waypoints": waypoints
#     }

# print("Funzione build_ondemand_vehicle pronta")

In [ ]:
#CELLA 1 — helper per rientro hub con timing preciso
# ══════════════════════════════════════════════════════════════════════════════
 
def t_to_hub_from(lat, lng):
    """Tempo di percorrenza da (lat,lng) a HUB."""
    n = find_nearest_node(lat, lng, node_latlon)
    node_hub = find_nearest_node(*HUB, node_latlon)
    t, _ = route_time_and_waypoints(G, node_latlon, n, node_hub, SPEED_MS)
    return t
 
def ensure_at_hub(wps, t_target):
    """
    Aggiunge waypoints per garantire che il veicolo sia a HUB esattamente a t_target.
    Se è già all'hub, aggiunge solo un punto di attesa.
    Restituisce i waypoints aggiornati e il t effettivo di arrivo all'hub.
    """
    last = wps[-1]
    at_hub = abs(last["lat"] - HUB[0]) < 0.0001 and abs(last["lng"] - HUB[1]) < 0.0001
    if at_hub:
        wps.append({"t": int(t_target), "lat": HUB[0], "lng": HUB[1]})
        return t_target
    t_travel = t_to_hub_from(last["lat"], last["lng"])
    t_depart = t_target - t_travel
    if t_depart > last["t"]:
        wps.append({"t": int(t_depart), "lat": last["lat"], "lng": last["lng"]})
    wps_home, _ = build_timed_waypoints(
        G, node_latlon, [(last["lat"], last["lng"]), HUB], max(t_depart, last["t"]), SPEED_MS
    )
    wps.extend(wps_home[1:])
    return t_target
 
print("Helper pronto")

In [ ]:
# CELLA 2 — costruisce un veicolo bacino (weekday / weekend)
# ══════════════════════════════════════════════════════════════════════════════
 
def build_bacino_vehicle(bacino, vehicle_idx, peak_am, peak_pm, seed_offset=0):
    """
    Struttura giornata:
      1. Mattina presto: hub → furthest → hub (rientro prima di peak_am)
      2. peak_am + idx*600: hub → pontedera → hub (turno staggered)
      3. On-demand nel bacino fino a peak_pm
      4. peak_pm + idx*600: hub → pontedera → hub (turno serale staggered)
      5. Sera: hub → furthest → hub
      6. On-demand fino a mezzanotte → rientro hub
    """
    rng = random.Random(RANDOM_SEED + seed_offset)
    stops    = bacini[bacino]
    furthest = stops[-1][:2]
    all_latlon = [s[:2] for s in stops]
 
    node_hub = find_nearest_node(*HUB, node_latlon)
    node_far = find_nearest_node(*furthest, node_latlon)
    t_h2f, _ = route_time_and_waypoints(G, node_latlon, node_hub, node_far, SPEED_MS)
    t_f2h, _ = route_time_and_waypoints(G, node_latlon, node_far, node_hub, SPEED_MS)
 
    wps = []
 
    # ── 1. MATTINA: hub → furthest → hub ─────────────────────────────────────
    t_depart_morning = peak_am - t_h2f - t_f2h - 300
    wps_to_far, t_at_far = build_timed_waypoints(
        G, node_latlon, [HUB, furthest], t_depart_morning, SPEED_MS
    )
    wps.extend(wps_to_far)
    wps_back, t_at_hub = build_timed_waypoints(
        G, node_latlon, [furthest, HUB], t_at_far, SPEED_MS
    )
    wps.extend(wps_back[1:])
 
    # ── 2. AM PONTEDERA: stagger 10 min ──────────────────────────────────────
    t_depart_am = peak_am + vehicle_idx * 600
    if t_at_hub < t_depart_am:
        wps.append({"t": int(t_depart_am), "lat": HUB[0], "lng": HUB[1]})
 
    wps_to_pont, t_at_pont = build_timed_waypoints(
        G, node_latlon, scheduled_coords, t_depart_am, SPEED_MS
    )
    wps.extend(wps_to_pont[1:])
 
    t_layover_am = t_at_pont + PONTEDERA_WAIT
    wps.append({"t": int(t_layover_am), "lat": scheduled_coords[-1][0], "lng": scheduled_coords[-1][1]})
 
    wps_from_pont, t_back_hub_am = build_timed_waypoints(
        G, node_latlon, list(reversed(scheduled_coords)), t_layover_am, SPEED_MS
    )
    wps.extend(wps_from_pont[1:])
 
    # ── 3. ON-DEMAND DIURNO ───────────────────────────────────────────────────
    t_depart_pm = peak_pm + vehicle_idx * 600
    t_limit_day = t_depart_pm - 600  # margine 10 min per rientrare all'hub
    t_current = t_back_hub_am
 
    while t_current < t_limit_day:
        wait = rng.randint(120, 300)
        t_current += wait
        if t_current >= t_limit_day:
            break
        last = wps[-1]
        n_sel = rng.randint(2, min(4, len(all_latlon)))
        selected_idx = sorted(rng.sample(range(len(all_latlon)), n_sel))
        selected = [all_latlon[j] for j in selected_idx]
        route = [(last["lat"], last["lng"])] + selected + [HUB]
        test_wps, t_end_run = build_timed_waypoints(G, node_latlon, route, t_current, SPEED_MS)
        if t_end_run >= t_limit_day:
            break
        wps.extend(test_wps[1:])
        t_current = t_end_run
 
    # ── 4. PM PONTEDERA: stagger 10 min ──────────────────────────────────────
    ensure_at_hub(wps, t_depart_pm)
 
    wps_to_pont_pm, t_at_pont_pm = build_timed_waypoints(
        G, node_latlon, scheduled_coords, t_depart_pm, SPEED_MS
    )
    wps.extend(wps_to_pont_pm[1:])
 
    t_layover_pm = t_at_pont_pm + PONTEDERA_WAIT
    wps.append({"t": int(t_layover_pm), "lat": scheduled_coords[-1][0], "lng": scheduled_coords[-1][1]})
 
    wps_from_pont_pm, t_back_hub_pm = build_timed_waypoints(
        G, node_latlon, list(reversed(scheduled_coords)), t_layover_pm, SPEED_MS
    )
    wps.extend(wps_from_pont_pm[1:])
 
    # ── 5. SERA: hub → furthest → hub ─────────────────────────────────────────
    wps_to_far_eve, t_at_far_eve = build_timed_waypoints(
        G, node_latlon, [HUB, furthest], t_back_hub_pm, SPEED_MS
    )
    wps.extend(wps_to_far_eve[1:])
    wps_back_eve, t_back_hub_eve = build_timed_waypoints(
        G, node_latlon, [furthest, HUB], t_at_far_eve, SPEED_MS
    )
    wps.extend(wps_back_eve[1:])
 
    # ── 6. ON-DEMAND NOTTURNO ─────────────────────────────────────────────────
    t_current = t_back_hub_eve
    t_limit_night = 86400 - 1200
 
    while t_current < t_limit_night:
        wait = rng.randint(120, 300)
        t_current += wait
        if t_current >= t_limit_night:
            break
        last = wps[-1]
        n_sel = rng.randint(2, min(4, len(all_latlon)))
        selected_idx = sorted(rng.sample(range(len(all_latlon)), n_sel))
        selected = [all_latlon[j] for j in selected_idx]
        route = [(last["lat"], last["lng"])] + selected + [HUB]
        test_wps, t_end_run = build_timed_waypoints(G, node_latlon, route, t_current, SPEED_MS)
        if t_end_run >= t_limit_night:
            break
        wps.extend(test_wps[1:])
        t_current = t_end_run
 
    # Rientro finale
    last = wps[-1]
    if abs(last["lat"] - HUB[0]) > 0.0001 or abs(last["lng"] - HUB[1]) > 0.0001:
        wps_final, _ = build_timed_waypoints(
            G, node_latlon, [(last["lat"], last["lng"]), HUB], t_current + 120, SPEED_MS
        )
        wps.extend(wps_final[1:])
 
    return {
        "id": bacino,
        "label": bacino.capitalize(),
        "color": COLORS[bacino],
        "type": "bacino",
        "waypoints": wps
    }
 
print("build_bacino_vehicle pronto")

In [ ]:
# CELLA 3 — costruisce il veicolo jolly (weekday / weekend)
# ══════════════════════════════════════════════════════════════════════════════
 
def build_jolly_vehicle(peak_am, peak_pm, all_stops_flat):
    """
    Jolly: fa giri random su tutte le fermate + pontedera.
    Rispetta i turni pontedera come il 6° veicolo (idx=5).
    """
    rng = random.Random(RANDOM_SEED + 99)
    JOLLY_IDX = 5
 
    t_depart_am = peak_am + JOLLY_IDX * 600
    t_depart_pm = peak_pm + JOLLY_IDX * 600
 
    wps = []
    t_current = t_depart_am - 3600
    wps.append({"t": int(t_current), "lat": HUB[0], "lng": HUB[1]})
 
    def random_route(t_start, t_limit):
        nonlocal t_current
        t = t_start
        while t < t_limit:
            wait = rng.randint(120, 300)
            t += wait
            if t >= t_limit:
                break
            last = wps[-1]
            n_sel = rng.randint(2, min(4, len(all_stops_flat)))
            selected = rng.sample(all_stops_flat, n_sel)
            route = [(last["lat"], last["lng"])] + selected + [HUB]
            test_wps, t_end_run = build_timed_waypoints(G, node_latlon, route, t, SPEED_MS)
            if t_end_run >= t_limit:
                break
            wps.extend(test_wps[1:])
            t = t_end_run
        return t
 
    # Mattina random fino al turno pontedera AM
    t_current = random_route(t_current, t_depart_am - 600)
    ensure_at_hub(wps, t_depart_am)
 
    # Pontedera AM
    wps_pont, t_at_pont = build_timed_waypoints(G, node_latlon, scheduled_coords, t_depart_am, SPEED_MS)
    wps.extend(wps_pont[1:])
    t_layover = t_at_pont + PONTEDERA_WAIT
    wps.append({"t": int(t_layover), "lat": scheduled_coords[-1][0], "lng": scheduled_coords[-1][1]})
    wps_back, t_back = build_timed_waypoints(G, node_latlon, list(reversed(scheduled_coords)), t_layover, SPEED_MS)
    wps.extend(wps_back[1:])
 
    # On-demand casuale fino al turno PM (può anche andare a pontedera — è già in all_stops_flat)
    t_current = random_route(t_back, t_depart_pm - 600)
    ensure_at_hub(wps, t_depart_pm)
 
    # Pontedera PM
    wps_pont_pm, t_at_pont_pm = build_timed_waypoints(G, node_latlon, scheduled_coords, t_depart_pm, SPEED_MS)
    wps.extend(wps_pont_pm[1:])
    t_layover_pm = t_at_pont_pm + PONTEDERA_WAIT
    wps.append({"t": int(t_layover_pm), "lat": scheduled_coords[-1][0], "lng": scheduled_coords[-1][1]})
    wps_back_pm, t_back_pm = build_timed_waypoints(G, node_latlon, list(reversed(scheduled_coords)), t_layover_pm, SPEED_MS)
    wps.extend(wps_back_pm[1:])
 
    # On-demand notturno
    t_current = random_route(t_back_pm, 86400 - 1200)
 
    # Rientro finale
    last = wps[-1]
    if abs(last["lat"] - HUB[0]) > 0.0001 or abs(last["lng"] - HUB[1]) > 0.0001:
        wps_final, _ = build_timed_waypoints(G, node_latlon, [(last["lat"], last["lng"]), HUB], t_current + 120, SPEED_MS)
        wps.extend(wps_final[1:])
 
    return {
        "id": "jolly",
        "label": "Jolly",
        "color": "#ffffff",
        "type": "jolly",
        "waypoints": wps
    }
 
print("build_jolly_vehicle pronto")

In [ ]:
# ── SCENARIO WEEKDAY ─────────────────────────────────────────────────────────
 
peak_am = WEEKDAY_PEAK_AM
peak_pm = WEEKDAY_PEAK_PM
 
# Tutte le fermate flat per il jolly
all_stops_flat = []
for b in ON_DEMAND_BACINI:
    all_stops_flat.extend([s[:2] for s in bacini[b]])
all_stops_flat.extend(scheduled_coords)
 
vehicles_weekday = []
for i, bacino in enumerate(ON_DEMAND_BACINI):
    if bacino not in bacini:
        continue
    v = build_bacino_vehicle(bacino, i, peak_am, peak_pm, seed_offset=i)
    vehicles_weekday.append(v)
    print(f"  ✓ {bacino}: {len(v['waypoints'])} waypoints")
 
jolly = build_jolly_vehicle(peak_am, peak_pm, all_stops_flat)
vehicles_weekday.append(jolly)
print(f"  ✓ jolly: {len(jolly['waypoints'])} waypoints")
 
weekday_scenario = {"scenario": "weekday", "vehicles": vehicles_weekday}
print(f"\nWeekday: {len(vehicles_weekday)} veicoli")

In [ ]:
# ── SCENARIO WEEKEND ─────────────────────────────────────────────────────────
 
peak_am = WEEKEND_PEAK_AM
peak_pm = WEEKEND_PEAK_PM
 
all_stops_flat_we = []
for b in ON_DEMAND_BACINI:
    all_stops_flat_we.extend([s[:2] for s in bacini[b]])
all_stops_flat_we.extend(scheduled_coords)
 
vehicles_weekend = []
for i, bacino in enumerate(ON_DEMAND_BACINI):
    if bacino not in bacini:
        continue
    v = build_bacino_vehicle(bacino, i, peak_am, peak_pm, seed_offset=i+10)
    vehicles_weekend.append(v)
    print(f"  ✓ {bacino}: {len(v['waypoints'])} waypoints")
 
jolly_we = build_jolly_vehicle(peak_am, peak_pm, all_stops_flat_we)
vehicles_weekend.append(jolly_we)
print(f"  ✓ jolly: {len(jolly_we['waypoints'])} waypoints")
 
weekend_scenario = {"scenario": "weekend", "vehicles": vehicles_weekend}
print(f"\nWeekend: {len(vehicles_weekend)} veicoli")

In [ ]:
# ── SCENARIO EVENTDAY ────────────────────────────────────────────────────────
# Struttura:
#   - Mattina uguale a weekday (furthest → hub prima di peak_am)
#   - AM pontedera staggered
#   - On-demand fino alle 12h
#   - 12h: tutti partono verso belvedere con gap 5 min (vehicle_idx * 300)
#   - Dopo ritorno da belvedere: on-demand nel bacino fino alle 17h
#   - 17h: tutti verso belvedere con gap 5 min, poi tornano a peccioli_1 con gap 10 min
#   - Dopo: on-demand fino a mezzanotte

peak_am = WEEKDAY_PEAK_AM
peak_pm = WEEKDAY_PEAK_PM
 
def build_bacino_eventday(bacino, vehicle_idx, peak_am, seed_offset=0):
    rng = random.Random(RANDOM_SEED + seed_offset)
    stops    = bacini[bacino]
    furthest = stops[-1][:2]
    all_latlon = [s[:2] for s in stops]
 
    node_hub = find_nearest_node(*HUB, node_latlon)
    node_far = find_nearest_node(*furthest, node_latlon)
    t_h2f, _ = route_time_and_waypoints(G, node_latlon, node_hub, node_far, SPEED_MS)
    t_f2h, _ = route_time_and_waypoints(G, node_latlon, node_far, node_hub, SPEED_MS)
 
    wps = []
 
    # ── 1. MATTINA: hub → furthest → hub ─────────────────────────────────────
    t_depart_morning = peak_am - t_h2f - t_f2h - 300
    wps_to_far, t_at_far = build_timed_waypoints(G, node_latlon, [HUB, furthest], t_depart_morning, SPEED_MS)
    wps.extend(wps_to_far)
    wps_back, t_at_hub = build_timed_waypoints(G, node_latlon, [furthest, HUB], t_at_far, SPEED_MS)
    wps.extend(wps_back[1:])
 
    # ── 2. AM PONTEDERA ───────────────────────────────────────────────────────
    t_depart_am = peak_am + vehicle_idx * 600
    if t_at_hub < t_depart_am:
        wps.append({"t": int(t_depart_am), "lat": HUB[0], "lng": HUB[1]})
    wps_pont, t_at_pont = build_timed_waypoints(G, node_latlon, scheduled_coords, t_depart_am, SPEED_MS)
    wps.extend(wps_pont[1:])
    t_lay = t_at_pont + PONTEDERA_WAIT
    wps.append({"t": int(t_lay), "lat": scheduled_coords[-1][0], "lng": scheduled_coords[-1][1]})
    wps_ret, t_back_am = build_timed_waypoints(G, node_latlon, list(reversed(scheduled_coords)), t_lay, SPEED_MS)
    wps.extend(wps_ret[1:])
 
    # ── 3. ON-DEMAND fino alle 12h ────────────────────────────────────────────
    T_EVENT1 = EVENT_TRIP1  # 12*3600
    t_depart_bel1 = T_EVENT1 + vehicle_idx * 300  # gap 5 min
    t_limit_pre_event = t_depart_bel1 - 600
    t_current = t_back_am
 
    while t_current < t_limit_pre_event:
        wait = rng.randint(120, 300)
        t_current += wait
        if t_current >= t_limit_pre_event:
            break
        last = wps[-1]
        n_sel = rng.randint(2, min(4, len(all_latlon)))
        selected_idx = sorted(rng.sample(range(len(all_latlon)), n_sel))
        selected = [all_latlon[j] for j in selected_idx]
        route = [(last["lat"], last["lng"])] + selected + [HUB]
        test_wps, t_end_run = build_timed_waypoints(G, node_latlon, route, t_current, SPEED_MS)
        if t_end_run >= t_limit_pre_event:
            break
        wps.extend(test_wps[1:])
        t_current = t_end_run
 
    # ── 4. EVENTO 12h: hub → belvedere_4 → hub ───────────────────────────────
    ensure_at_hub(wps, t_depart_bel1)
    wps_bel1, t_at_bel1 = build_timed_waypoints(G, node_latlon, [HUB, BEL5], t_depart_bel1, SPEED_MS)
    wps.extend(wps_bel1[1:])
    wps_ret_bel1, t_back_bel1 = build_timed_waypoints(G, node_latlon, [BEL5, HUB], t_at_bel1 + 300, SPEED_MS)
    wps.extend(wps_ret_bel1[1:])
 
    # ── 5. ON-DEMAND 12h-17h nel bacino ──────────────────────────────────────
    T_EVENT2 = EVENT_TRIP2  # 17*3600
    t_depart_bel2 = T_EVENT2 + vehicle_idx * 300
    t_limit_mid_event = t_depart_bel2 - 600
    t_current = t_back_bel1
 
    while t_current < t_limit_mid_event:
        wait = rng.randint(120, 300)
        t_current += wait
        if t_current >= t_limit_mid_event:
            break
        last = wps[-1]
        n_sel = rng.randint(2, min(4, len(all_latlon)))
        selected_idx = sorted(rng.sample(range(len(all_latlon)), n_sel))
        selected = [all_latlon[j] for j in selected_idx]
        route = [(last["lat"], last["lng"])] + selected + [HUB]
        test_wps, t_end_run = build_timed_waypoints(G, node_latlon, route, t_current, SPEED_MS)
        if t_end_run >= t_limit_mid_event:
            break
        wps.extend(test_wps[1:])
        t_current = t_end_run
 
    # ── 6. EVENTO 17h: hub → belvedere_4 ─────────────────────────────────────
    ensure_at_hub(wps, t_depart_bel2)
    wps_bel2, t_at_bel2 = build_timed_waypoints(G, node_latlon, [HUB, BEL5], t_depart_bel2, SPEED_MS)
    wps.extend(wps_bel2[1:])
 
    # Ritorno da belvedere con gap 10 min
    t_depart_home = t_at_bel2 + vehicle_idx * 600
    wps.append({"t": int(t_depart_home), "lat": BEL5[0], "lng": BEL5[1]})
    wps_home, t_back_home = build_timed_waypoints(G, node_latlon, [BEL5, HUB], t_depart_home, SPEED_MS)
    wps.extend(wps_home[1:])
 
    # ── 7. ON-DEMAND NOTTURNO fino a mezzanotte ───────────────────────────────
    t_current = t_back_home
    t_limit_night = 86400 - 1200
 
    while t_current < t_limit_night:
        wait = rng.randint(120, 300)
        t_current += wait
        if t_current >= t_limit_night:
            break
        last = wps[-1]
        n_sel = rng.randint(2, min(4, len(all_latlon)))
        selected_idx = sorted(rng.sample(range(len(all_latlon)), n_sel))
        selected = [all_latlon[j] for j in selected_idx]
        route = [(last["lat"], last["lng"])] + selected + [HUB]
        test_wps, t_end_run = build_timed_waypoints(G, node_latlon, route, t_current, SPEED_MS)
        if t_end_run >= t_limit_night:
            break
        wps.extend(test_wps[1:])
        t_current = t_end_run
 
    last = wps[-1]
    if abs(last["lat"] - HUB[0]) > 0.0001 or abs(last["lng"] - HUB[1]) > 0.0001:
        wps_final, _ = build_timed_waypoints(G, node_latlon, [(last["lat"], last["lng"]), HUB], t_current + 120, SPEED_MS)
        wps.extend(wps_final[1:])
 
    return {
        "id": bacino,
        "label": bacino.capitalize(),
        "color": COLORS[bacino],
        "type": "bacino",
        "waypoints": wps
    }
 
 
def build_jolly_eventday(peak_am, all_stops_flat):
    """Jolly nell'evento: segue la struttura dei bacini ma con fermate random ovunque."""
    rng = random.Random(RANDOM_SEED + 99)
    JOLLY_IDX = 5
    T_EVENT1 = EVENT_TRIP1
    T_EVENT2 = EVENT_TRIP2
 
    t_depart_am   = peak_am + JOLLY_IDX * 600
    t_depart_bel1 = T_EVENT1 + JOLLY_IDX * 300
    t_depart_bel2 = T_EVENT2 + JOLLY_IDX * 300
 
    wps = []
    t_current = t_depart_am - 3600
    wps.append({"t": int(t_current), "lat": HUB[0], "lng": HUB[1]})
 
    def random_route_to(t_limit):
        nonlocal t_current
        while t_current < t_limit:
            wait = rng.randint(120, 300)
            t_current += wait
            if t_current >= t_limit:
                break
            last = wps[-1]
            n_sel = rng.randint(2, min(4, len(all_stops_flat)))
            selected = rng.sample(all_stops_flat, n_sel)
            route = [(last["lat"], last["lng"])] + selected + [HUB]
            test_wps, t_end_run = build_timed_waypoints(G, node_latlon, route, t_current, SPEED_MS)
            if t_end_run >= t_limit:
                break
            wps.extend(test_wps[1:])
            t_current = t_end_run
 
    # Pre-pontedera
    random_route_to(t_depart_am - 600)
    ensure_at_hub(wps, t_depart_am)
 
    # Pontedera AM
    wps_pont, t_at_pont = build_timed_waypoints(G, node_latlon, scheduled_coords, t_depart_am, SPEED_MS)
    wps.extend(wps_pont[1:])
    t_lay = t_at_pont + PONTEDERA_WAIT
    wps.append({"t": int(t_lay), "lat": scheduled_coords[-1][0], "lng": scheduled_coords[-1][1]})
    wps_ret, t_back_am = build_timed_waypoints(G, node_latlon, list(reversed(scheduled_coords)), t_lay, SPEED_MS)
    wps.extend(wps_ret[1:])
    t_current = t_back_am
 
    # On-demand fino all'evento 12h
    random_route_to(t_depart_bel1 - 600)
    ensure_at_hub(wps, t_depart_bel1)
 
    # Belvedere 12h
    wps_bel1, t_at_bel1 = build_timed_waypoints(G, node_latlon, [HUB, BEL5], t_depart_bel1, SPEED_MS)
    wps.extend(wps_bel1[1:])
    wps_rb1, t_rb1 = build_timed_waypoints(G, node_latlon, [BEL5, HUB], t_at_bel1 + 300, SPEED_MS)
    wps.extend(wps_rb1[1:])
    t_current = t_rb1
 
    # On-demand 12h-17h
    random_route_to(t_depart_bel2 - 600)
    ensure_at_hub(wps, t_depart_bel2)
 
    # Belvedere 17h
    wps_bel2, t_at_bel2 = build_timed_waypoints(G, node_latlon, [HUB, BEL5], t_depart_bel2, SPEED_MS)
    wps.extend(wps_bel2[1:])
    t_depart_home = t_at_bel2 + JOLLY_IDX * 600
    wps.append({"t": int(t_depart_home), "lat": BEL5[0], "lng": BEL5[1]})
    wps_home, t_back_home = build_timed_waypoints(G, node_latlon, [BEL5, HUB], t_depart_home, SPEED_MS)
    wps.extend(wps_home[1:])
    t_current = t_back_home
 
    # Notturno
    random_route_to(86400 - 1200)
 
    last = wps[-1]
    if abs(last["lat"] - HUB[0]) > 0.0001 or abs(last["lng"] - HUB[1]) > 0.0001:
        wps_final, _ = build_timed_waypoints(G, node_latlon, [(last["lat"], last["lng"]), HUB], t_current + 120, SPEED_MS)
        wps.extend(wps_final[1:])
 
    return {
        "id": "jolly",
        "label": "Jolly",
        "color": "#ffffff",
        "type": "jolly",
        "waypoints": wps
    }
 
 
# Build eventday
all_stops_flat_ev = []
for b in ON_DEMAND_BACINI:
    all_stops_flat_ev.extend([s[:2] for s in bacini[b]])
all_stops_flat_ev.extend(scheduled_coords)
 
vehicles_event = []
for i, bacino in enumerate(ON_DEMAND_BACINI):
    if bacino not in bacini:
        continue
    v = build_bacino_eventday(bacino, i, peak_am, seed_offset=i+20)
    vehicles_event.append(v)
    print(f"  ✓ {bacino}: {len(v['waypoints'])} waypoints (evento)")
 
jolly_ev = build_jolly_eventday(peak_am, all_stops_flat_ev)
vehicles_event.append(jolly_ev)
print(f"  ✓ jolly: {len(jolly_ev['waypoints'])} waypoints (evento)")
 
eventday_scenario = {"scenario": "eventday", "vehicles": vehicles_event}
print(f"\nEventday: {len(vehicles_event)} veicoli")

In [ ]:
# ── ESPORTA JSON ─────────────────────────────────────────────────────────────

import os
os.makedirs(OUT_DIR, exist_ok=True)

for scenario, fname in [
    (weekday_scenario,  "weekday.json"),
    (weekend_scenario,  "weekend.json"),
    (eventday_scenario, "eventday.json"),
]:
    path = os.path.join(OUT_DIR, fname)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(scenario, f, ensure_ascii=False, separators=(",", ":"))
    size_kb = os.path.getsize(path) / 1024
    print(f"  ✓ {fname} — {size_kb:.1f} KB")

print("\n✅ Tutti gli scenari esportati. Copia la cartella data/scenarios/ nel repo GitHub.")

In [ ]:
# ── DEBUG: visualizza waypoints di un veicolo ────────────────────────────────
# Cambia il nome del veicolo per ispezionare

VEH_TO_INSPECT = "rosa"  # id del veicolo
SCENARIO = weekday_scenario

veh = next((v for v in SCENARIO["vehicles"] if v["id"] == VEH_TO_INSPECT), None)
if veh:
    print(f"Veicolo: {veh['id']} | {len(veh['waypoints'])} waypoints")
    print(f"Inizio: {veh['waypoints'][0]}")
    print(f"Fine:   {veh['waypoints'][-1]}")
    for wp in veh['waypoints'][:5]:
        t = wp['t']
        print(f"  t={t//3600:02d}:{(t%3600)//60:02d}:{t%60:02d}  lat={wp['lat']:.5f}  lng={wp['lng']:.5f}")
    print("  ...")
else:
    print(f"Veicolo '{VEH_TO_INSPECT}' non trovato")

In [ ]:
# ── ESPORTA GRAFO COME GEOJSON ───────────────────────────────────────────────
import os
network_out = network_gdf[["geometry"]].to_crs(epsg=4326)
network_out.to_file(
    os.path.join(OUT_DIR, "../network.geojson"),
    driver="GeoJSON"
)
print("✓ network.geojson esportato")